# 04 - LangGraph: Remediation & Human-in-the-Loop

## Scenario: Northstar Server Remediation

In Module 02, we built a raw agent loop. We saw that if a tool is dangerous (like `restart_server`), we can't let the agent execute it automatically. We need a **Human-in-the-Loop (HITL)**. 

While you *can* build HITL manually in a raw loop, it's difficult to pause the loop, persist the state, and resume it days later. This is where **LangGraph** excels. It models agent states as graphs and natively supports breakpoints.

In this notebook, we will build a Northstar Remediation Agent that investigates servers and stops to ask for approval before restarting them.

In [1]:
import os
import json
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# 1. Initialize LLM
llm = ChatOpenAI(model="gpt-4o", api_key=os.environ.get("OPENAI_API_KEY", "dummy-key"))

# 2. Define Tools
@tool
def check_server(server_id: str) -> str:
    """Check if a server is healthy."""
    print(f"  🔧 [Tool Executing] Checking {server_id}...")
    return "Server is failing with 99% CPU."

@tool
def restart_server(server_id: str) -> str:
    """Restart the server. Requires human approval!"""
    print(f"  🚨 [Tool Executing] RESTARTING {server_id}...")
    return f"{server_id} restarted successfully."

tools = [check_server, restart_server]
llm_with_tools = llm.bind_tools(tools)


/Users/mahsateimourikia/repos/awsome-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Defining the Graph State

LangGraph uses a `TypedDict` to pass state between nodes. The most important part of an Agent's state is its message history.

In [2]:
class AgentState(TypedDict):
    # `add_messages` ensures new messages are appended to the list, not overwritten
    messages: Annotated[list[BaseMessage], add_messages]


## 2. Defining the Nodes and Edges

We need two nodes:
1. **Agent Node**: Calls the LLM to decide the next action.
2. **Action Node**: Executes the tools using LangGraph's prebuilt `ToolNode`.

In [3]:
# Define Agent Node
def call_model(state: AgentState):
    print("-> Agent is thinking...")
    # Mock LLM for the sake of the tutorial if API key is missing
    try:
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}
    except Exception:
        # Fallback Mock Logic
        from langchain_core.messages import AIMessage
        from langchain_core.messages.tool import ToolCall
        
        last_msg = state["messages"][-1].content
        if "failing" in last_msg:
            print("-> Agent (Mock): Wants to restart server.")
            return {"messages": [AIMessage(content="", tool_calls=[ToolCall(name="restart_server", args={"server_id": "eu-web-01"}, id="call_abc")])]}
        else:
            print("-> Agent (Mock): Wants to check server.")
            return {"messages": [AIMessage(content="", tool_calls=[ToolCall(name="check_server", args={"server_id": "eu-web-01"}, id="call_123")])]}

# Define Routing Edge
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    # If there is no tool call, the agent has finished
    if not last_message.tool_calls:
        return "__end__"
    return "tools"

# Build Graph
builder = StateGraph(AgentState)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))

builder.set_entry_point("agent")
builder.add_conditional_edges("agent", should_continue)
builder.add_edge("tools", "agent")

# Compile with a Checkpointer (Memory) and a BREAKPOINT before the tools node
memory = MemorySaver()
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"]  # <--- HUMAN IN THE LOOP BREAKPOINT
)


## 3. Running with Human-in-the-Loop

Let's start an investigation. We pass a `thread_id` so LangGraph remembers this specific conversation in its memory checkpointer.

In [4]:
thread = {"configurable": {"thread_id": "incident-402"}}
inputs = {"messages": [HumanMessage(content="Investigate server eu-web-01.")]}

print("--- Starting Graph ---")
for event in graph.stream(inputs, thread, stream_mode="values"):
    pass

# We expect the graph to PAUSE because it wants to call the check_server tool, 
# and we told it to interrupt_before=["tools"].
print("\nGraph State after first run:")
state = graph.get_state(thread)
print(f"Next node to run: {state.next}")


--- Starting Graph ---
-> Agent is thinking...
-> Agent (Mock): Wants to check server.

Graph State after first run:
Next node to run: ('tools',)


Because it paused, we must explicitly approve the continuation.

In [5]:
print("\n--- Approving execution of tools ---")
# Passing None resumes execution from the breakpoint
for event in graph.stream(None, thread, stream_mode="values"):
    pass

print("\nGraph State after resuming:")
state = graph.get_state(thread)
print(f"Next node to run: {state.next}")



--- Approving execution of tools ---
  🔧 [Tool Executing] Checking eu-web-01...
-> Agent is thinking...
-> Agent (Mock): Wants to restart server.

Graph State after resuming:
Next node to run: ('tools',)


If the agent discovered the server was failing, it probably wants to run `restart_server` now. The graph pauses again! This gives the human operator a chance to review the dangerous tool call.

In [6]:
last_message = graph.get_state(thread).values["messages"][-1]
if last_message.tool_calls:
    print(f"\n🚨 Agent wants to execute: {last_message.tool_calls[0]['name']}")
    print("Do you approve? (Simulating YES)")
    
    # Approve the restart
    for event in graph.stream(None, thread, stream_mode="values"):
        pass



🚨 Agent wants to execute: restart_server
Do you approve? (Simulating YES)
  🚨 [Tool Executing] RESTARTING eu-web-01...
-> Agent is thinking...


-> Agent (Mock): Wants to check server.


## Watch For

- **Memory Savers in Production**: We used an in-memory `MemorySaver`. In production, you must use a database backend (like PostgreSQL) so the thread state persists even if the pod restarts.
- **Interrupt Granularity**: `interrupt_before=["tools"]` interrupts *every* tool call. In a real system, you might want to only interrupt specific dangerous tools. You can do this by routing dangerous tools to a separate `dangerous_tools_node` and only interrupting that node.

## Checkpoint

**1. What does `interrupt_before=["tools"]` do in LangGraph?**
- A) It prevents the agent from ever using tools.
- B) It deletes the tools from the agent's memory.
- C) It pauses the graph execution right before the `tools` node runs, allowing a human or external system to inspect the state and approve continuation.
- D) It causes an exception if tools take too long to run.

**2. Why is a `checkpointer` (like `MemorySaver`) required for Human-in-the-Loop workflows?**
- A) To save OpenAI API keys securely.
- B) Because pausing a graph means the application might exit. The checkpointer persists the current state (like variables and message history) so the graph can be resumed later.
- C) To make the graph run faster.
- D) To prevent hallucinations.
